In [16]:
corpus = [
    "virat scored century in match",
    "bjp won in elections",
    "Bumra took 5 wicket in a match",
    "Congress form state government"
]


In [ ]:
documents = corpus
print(documents)


['virat scored century in match', 'bjp won in elections', 'Bumra took 5 wicket in a match', 'Congress form state government']


In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

processed_docs = []

for doc in documents:

    # Clean text (remove numbers & special chars)
    doc = re.sub(r'[^a-zA-Z]', ' ', doc)

    # Lowercase
    doc = doc.lower()

    # Tokenization
    words = doc.split()

    # Stopword removal + Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # Rejoin
    processed_docs.append(" ".join(words))

print(processed_docs)

['virat scored century match', 'bjp election', 'bumra took wicket match', 'congress form state government']


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(processed_docs)


In [ ]:
import pandas as pd

bow_df = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())
print(bow_df)

   bjp  bumra  century  congress  election  form  government  match  scored  \
0    0      0        1         0         0     0           0      1       1   
1    1      0        0         0         1     0           0      0       0   
2    0      1        0         0         0     0           0      1       0   
3    0      0        0         1         0     1           1      0       0   

   state  took  virat  wicket  
0      0     0      1       0  
1      0     0      0       0  
2      0     1      0       1  
3      1     0      0       0  


In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(bow)


LatentDirichletAllocation(n_components=2, random_state=42)

In [ ]:
words = vectorizer.get_feature_names_out()

for i, topic in enumerate(lda.components_):
    print(f"\nTopic {i+1}:")
    print([words[index] for index in topic.argsort()[-5:]])



Topic 1:
['election', 'state', 'congress', 'government', 'form']

Topic 2:
['took', 'scored', 'century', 'virat', 'match']


In [ ]:
#task-2

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

import nltk
from nltk.corpus import stopwords
import re


In [19]:
df = pd.read_csv("/content/arxiv_data.csv", engine='python', on_bad_lines='skip') # change dataset name

print(df.head())

                                              titles  \
0  Survey on Semantic Stereo Matching / Semantic ...   
1  FUTURE-AI: Guiding Principles and Consensus Re...   
2  Enforcing Mutual Consistency of Hard Regions f...   
3  Parameter Decoupling Strategy for Semi-supervi...   
4  Background-Foreground Segmentation for Interio...   

                                           summaries  \
0  Stereo matching is one of the widely used tech...   
1  The recent advancements in artificial intellig...   
2  In this paper, we proposed a novel mutual cons...   
3  Consistency training has proven to be an advan...   
4  To ensure safety in automated driving, the cor...   

                         terms  
0           ['cs.CV', 'cs.LG']  
1  ['cs.CV', 'cs.AI', 'cs.LG']  
2           ['cs.CV', 'cs.AI']  
3                    ['cs.CV']  
4           ['cs.CV', 'cs.LG']  


In [20]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Ensure text is a string before processing
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

# Assuming you want to clean the 'summaries' column from the df DataFrame
# texts = df['summaries'].apply(clean_text) # Original line
texts = df['summaries'].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

texts_clean = texts.apply(preprocess)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')
bow_matrix = vectorizer.fit_transform(texts_clean)


In [22]:
print(texts_clean.isnull().sum())
print(texts_clean.head())


0
0    stereo matching one widely used techniques inf...
1    recent advancements artificial intelligence ai...
2    paper proposed novel mutual consistency networ...
3    consistency training proven advanced semisuper...
4    ensure safety automated driving correct percep...
Name: summaries, dtype: object


In [23]:
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print(bow_df.head())


   aa  aafor  aasegfor  aatr  abackgroundwhich  abalanced  aball  abandonthe  \
0   0      0         0     0                 0          0      0           0   
1   0      0         0     0                 0          0      0           0   
2   0      0         0     0                 0          0      0           0   
3   0      0         0     0                 0          0      0           0   
4   0      0         0     0                 0          0      0           0   

   abasic  abatch  ...  znn  zoclip  zoclipto  zones  zoom  zooming  zoonotic  \
0       0       0  ...    0       0         0      0     0        0         0   
1       0       0  ...    0       0         0      0     0        0         0   
2       0       0  ...    0       0         0      0     0        0         0   
3       0       0  ...    0       0         0      0     0        0         0   
4       0       0  ...    0       0         0      0     0        0         0   

   zoos  zsl  zurich  
0     0  

In [24]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(
    n_components=5,   # number of topics
    random_state=42
)

lda_model.fit(bow_matrix)


LatentDirichletAllocation(n_components=5, random_state=42)

In [25]:
def display_topics(model, feature_names, no_words):
    for topic_idx, topic in enumerate(model.components_):
        print("\nTopic", topic_idx+1)
        print([feature_names[i] for i in topic.argsort()[:-no_words - 1:-1]])

display_topics(lda_model, vectorizer.get_feature_names_out(), 10)



Topic 1
['segmentation', 'image', 'images', 'method', 'model', 'data', 'learning', 'proposed', 'results', 'network']

Topic 2
['learning', 'graph', 'data', 'representations', 'representation', 'tasks', 'model', 'propose', 'information', 'methods']

Topic 3
['learning', 'segmentation', 'image', 'representation', 'methods', 'method', 'images', 'network', 'deep', 'propose']

Topic 4
['segmentation', 'image', 'learning', 'data', 'model', 'method', 'images', 'results', 'models', 'performance']

Topic 5
['learning', 'data', 'image', 'segmentation', 'training', 'model', 'images', 'method', 'methods', 'using']


In [26]:
#task-3

In [27]:
import pandas as pd

# Corpus
documents = [
    "Virat scored century in match",
    "BJP won in elections",
    "Bumra took 5 wicket in a match",
    "Congress form state government"
]

df = pd.DataFrame(documents, columns=["Text"])
print(df)


                             Text
0   Virat scored century in match
1            BJP won in elections
2  Bumra took 5 wicket in a match
3  Congress form state government


In [28]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added to download the missing resource

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercase
    text = text.lower()

    # Remove numbers & punctuation
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenize
    words = word_tokenize(text)

    # Remove stopwords & Lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]

    # Rejoin
    return " ".join(words)

df["Clean_Text"] = df["Text"].apply(preprocess)

print(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                             Text                      Clean_Text
0   Virat scored century in match      virat scored century match
1            BJP won in elections                    bjp election
2  Bumra took 5 wicket in a match         bumra took wicket match
3  Congress form state government  congress form state government


In [29]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(df["Clean_Text"])



In [30]:
bow_df = pd.DataFrame(bow.toarray(), columns=vectorizer.get_feature_names_out())
print(bow_df)


   bjp  bumra  century  congress  election  form  government  match  scored  \
0    0      0        1         0         0     0           0      1       1   
1    1      0        0         0         1     0           0      0       0   
2    0      1        0         0         0     0           0      1       0   
3    0      0        0         1         0     1           1      0       0   

   state  took  virat  wicket  
0      0     0      1       0  
1      0     0      0       0  
2      0     1      0       1  
3      1     0      0       0  


In [31]:
from sklearn.decomposition import NMF

num_topics = 2

nmf_model = NMF(n_components=num_topics, random_state=42)
nmf_model.fit(bow)


NMF(n_components=2, random_state=42)

In [32]:
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    print(f"\nTopic {topic_idx + 1}:")
    print([feature_names[i] for i in topic.argsort()[-5:]])



Topic 1:
['wicket', 'scored', 'took', 'virat', 'match']

Topic 2:
['election', 'government', 'congress', 'form', 'state']


In [33]:
topic_values = nmf_model.transform(bow)

df["Topic"] = topic_values.argmax(axis=1) + 1

print("\nFinal Topic Assignment:")
print(df[["Text", "Topic"]])


Final Topic Assignment:
                             Text  Topic
0   Virat scored century in match      1
1            BJP won in elections      2
2  Bumra took 5 wicket in a match      1
3  Congress form state government      2


In [ ]:
#task-4

In [34]:
import pandas as pd

# Load dataset (change path to your downloaded Kaggle file)
df = pd.read_csv("/content/arxiv_data.csv", on_bad_lines='warn', engine='python')   # example file name

# Display columns
print(df.columns)

# Assume abstract column contains text
corpus = df['summaries'].dropna().tolist()

print("Total Documents:", len(corpus))

Index(['titles', 'summaries', 'terms'], dtype='object')
Total Documents: 2383


In [35]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercase
    text = text.lower()

    # Remove numbers & special characters
    text = re.sub(r'[^a-zA-Z ]', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Rejoin
    return " ".join(tokens)

clean_corpus = [preprocess(doc) for doc in corpus]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [36]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=3000)

bow_matrix = vectorizer.fit_transform(clean_corpus)


In [37]:
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

print(bow_df.head())


   abdominal  ability  ablation  able  abnormal  abnormality  absence  \
0          0        0         0     0         0            0        0   
1          0        0         0     0         0            0        0   
2          0        0         0     0         0            0        0   
3          0        1         0     0         0            0        0   
4          0        0         0     0         0            0        0   

   absolute  abstract  abstraction  ...  workwe  world  would  xray  year  \
0         0         0            0  ...       0      0      0     0     0   
1         0         0            0  ...       0      0      0     0     0   
2         0         0            0  ...       0      0      0     0     0   
3         0         0            0  ...       0      0      1     0     0   
4         0         0            0  ...       0      0      0     0     1   

   yet  yield  yielded  zero  zeroshot  
0    0      0        0     0         0  
1    0      0   

In [38]:
from sklearn.decomposition import NMF

num_topics = 5

nmf_model = NMF(n_components=num_topics, random_state=42)
nmf_model.fit(bow_matrix)


NMF(n_components=5, random_state=42)

In [39]:
feature_names = vectorizer.get_feature_names_out()

def display_topics(model, feature_names, num_words=10):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx+1}:")
        print([feature_names[i] for i in topic.argsort()[:-num_words-1:-1]])

display_topics(nmf_model, feature_names)




Topic 1:
['image', 'segmentation', 'method', 'algorithm', 'based', 'result', 'proposed', 'region', 'approach', 'using']

Topic 2:
['learning', 'representation', 'task', 'data', 'method', 'feature', 'contrastive', 'deep', 'propose', 'network']

Topic 3:
['segmentation', 'network', 'method', 'deep', 'task', 'neural', 'medical', 'performance', 'result', 'semantic']

Topic 4:
['graph', 'node', 'network', 'method', 'representation', 'information', 'structure', 'edge', 'embedding', 'propose']

Topic 5:
['model', 'data', 'training', 'domain', 'show', 'performance', 'prediction', 'trained', 'deep', 'datasets']


In [41]:
topic_values = nmf_model.transform(bow_matrix)

# Filter df to match the length of topic_values (which was derived from non-null summaries)
df_filtered = df.dropna(subset=['summaries']).copy()

df_filtered['Dominant_Topic'] = topic_values.argmax(axis=1)

print(df_filtered[['summaries', 'Dominant_Topic']].head())

                                           summaries  Dominant_Topic
0  Stereo matching is one of the widely used tech...               2
1  The recent advancements in artificial intellig...               0
2  In this paper, we proposed a novel mutual cons...               4
3  Consistency training has proven to be an advan...               4
4  To ensure safety in automated driving, the cor...               1


In [42]:
#Task-5(sample data-1 using TF.IDF)

In [43]:
import pandas as pd

# Load Kaggle dataset (change file name/path)
df = pd.read_csv("/content/arxiv_data.csv", on_bad_lines='warn', engine='python')

# Check columns
print(df.columns)

# Take abstracts as corpus
corpus = df['summaries'].dropna().tolist()

print("Total Documents:", len(corpus))

Index(['titles', 'summaries', 'terms'], dtype='object')
Total Documents: 2383


In [44]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z ]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal + Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Rejoin
    return " ".join(tokens)

clean_corpus = [preprocess(doc) for doc in corpus]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=3000)

tfidf_matrix = tfidf_vectorizer.fit_transform(clean_corpus)




In [46]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print(tfidf_df.head())


   abdominal   ability  ablation  able  abnormal  abnormality  absence  \
0        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
1        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
2        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
3        0.0  0.082864       0.0   0.0       0.0          0.0      0.0   
4        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   

   absolute  abstract  abstraction  ...  workwe  world     would  xray  \
0       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
1       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
2       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
3       0.0       0.0          0.0  ...     0.0    0.0  0.110243   0.0   
4       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   

       year  yet  yield  yielded  zero  zeroshot  
0  0.000000  0.0    0.0      0.0   0.0       0.0  
1  0.000

In [47]:
from sklearn.decomposition import LatentDirichletAllocation

num_topics = 5

lda_model = LatentDirichletAllocation(
    n_components=num_topics,
    random_state=42
)

lda_model.fit(tfidf_matrix)

LatentDirichletAllocation(n_components=5, random_state=42)

In [48]:

feature_names = tfidf_vectorizer.get_feature_names_out()

def display_topics(model, feature_names, num_words=10):

    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx+1}:")
        print([feature_names[i] for i in topic.argsort()[:-num_words-1:-1]])

display_topics(lda_model, feature_names)



Topic 1:
['root', 'fullyconnected', 'bandit', 'smoothly', 'artist', 'painting', 'dml', 'contact', 'coding', 'bilevel']

Topic 2:
['learning', 'representation', 'graph', 'model', 'data', 'task', 'method', 'image', 'network', 'feature']

Topic 3:
['segmentation', 'image', 'method', 'network', 'algorithm', 'model', 'medical', 'proposed', 'approach', 'result']

Topic 4:
['efis', 'gt', 'biofilm', 'switching', 'evolving', 'curvature', 'coverage', 'adjustment', 'enforcing', 'tooth']

Topic 5:
['gnns', 'molecular', 'molecule', 'hyperbolic', 'message', 'hypergraph', 'sketch', 'chemical', 'fss', 'passing']


In [50]:
topic_values = lda_model.transform(tfidf_matrix)

# Filter df to match the length of topic_values (which was derived from non-null summaries)
df_filtered = df.dropna(subset=['summaries']).copy()

df_filtered['Dominant_Topic'] = topic_values.argmax(axis=1)

print(df_filtered[['summaries', 'Dominant_Topic']].head())

                                           summaries  Dominant_Topic
0  Stereo matching is one of the widely used tech...               2
1  The recent advancements in artificial intellig...               2
2  In this paper, we proposed a novel mutual cons...               1
3  Consistency training has proven to be an advan...               1
4  To ensure safety in automated driving, the cor...               2


In [51]:
#task-6(using nmf)

In [52]:

import pandas as pd

# Load Kaggle dataset (change file name/path)
df = pd.read_csv("/content/arxiv_data.csv", on_bad_lines='warn', engine='python')

# Check columns
print(df.columns)

# Take abstracts as corpus
corpus = df['summaries'].dropna().tolist()

print("Total Documents:", len(corpus))


Index(['titles', 'summaries', 'terms'], dtype='object')
Total Documents: 2383


In [53]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    # Lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z ]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal + Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Rejoin
    return " ".join(tokens)

clean_corpus = [preprocess(doc) for doc in corpus]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [54]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=3000)

bow_matrix = vectorizer.fit_transform(clean_corpus)



tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print(tfidf_df.head())

   abdominal   ability  ablation  able  abnormal  abnormality  absence  \
0        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
1        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
2        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   
3        0.0  0.082864       0.0   0.0       0.0          0.0      0.0   
4        0.0  0.000000       0.0   0.0       0.0          0.0      0.0   

   absolute  abstract  abstraction  ...  workwe  world     would  xray  \
0       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
1       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
2       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   
3       0.0       0.0          0.0  ...     0.0    0.0  0.110243   0.0   
4       0.0       0.0          0.0  ...     0.0    0.0  0.000000   0.0   

       year  yet  yield  yielded  zero  zeroshot  
0  0.000000  0.0    0.0      0.0   0.0       0.0  
1  0.000

In [55]:

from sklearn.decomposition import LatentDirichletAllocation

num_topics = 5

lda_model = LatentDirichletAllocation(
    n_components=num_topics,
    random_state=42
)

lda_model.fit(tfidf_matrix)


LatentDirichletAllocation(n_components=5, random_state=42)

In [56]:
feature_names = tfidf_vectorizer.get_feature_names_out()

def display_topics(model, feature_names, num_words=10):

    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx+1}:")
        print([feature_names[i] for i in topic.argsort()[:-num_words-1:-1]])

display_topics(lda_model, feature_names)


Topic 1:
['root', 'fullyconnected', 'bandit', 'smoothly', 'artist', 'painting', 'dml', 'contact', 'coding', 'bilevel']

Topic 2:
['learning', 'representation', 'graph', 'model', 'data', 'task', 'method', 'image', 'network', 'feature']

Topic 3:
['segmentation', 'image', 'method', 'network', 'algorithm', 'model', 'medical', 'proposed', 'approach', 'result']

Topic 4:
['efis', 'gt', 'biofilm', 'switching', 'evolving', 'curvature', 'coverage', 'adjustment', 'enforcing', 'tooth']

Topic 5:
['gnns', 'molecular', 'molecule', 'hyperbolic', 'message', 'hypergraph', 'sketch', 'chemical', 'fss', 'passing']
